# Model Inference — Walmart Store Sales Forecasting

In [1]:
try:
    from google.colab import drive
    import os
    drive.mount("/content/drive")
    os.environ["WALMART_ROOT"] = "/content/drive/MyDrive/MLFinalAssignment"
except ImportError:
    pass

Mounted at /content/drive


In [ ]:
import importlib.util
import os
import pathlib
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    missing = [p for p in ("lightgbm", "mlflow", "dagshub")
               if importlib.util.find_spec(p) is None]
    if missing:
        print("installing", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    if not pathlib.Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")


def find_root() -> pathlib.Path:
    """Locate the repo: it must hold train.csv and src/walmart_prep.py."""
    candidates = []
    if os.environ.get("WALMART_ROOT"):
        candidates.append(pathlib.Path(os.environ["WALMART_ROOT"]))
    candidates += [pathlib.Path.cwd(), pathlib.Path.cwd().parent]
    if IN_COLAB:
        drive_root = pathlib.Path("/content/drive/MyDrive")
        candidates += [drive_root / "MLFinalProject", pathlib.Path("/content/MLFinalProject")]
        if drive_root.exists():
            candidates += sorted(p for p in drive_root.glob("*") if (p / "train.csv").exists())
    for c in candidates:
        if (c / "train.csv").exists() and (c / "src" / "walmart_prep.py").exists():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find the project. Set WALMART_ROOT to the folder containing "
        "train.csv and src/walmart_prep.py, then re-run this cell.\n"
        f"Looked in: {[str(c) for c in candidates]}")


ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
for sub in ("docs", "submissions"):
    (ROOT / sub).mkdir(exist_ok=True)

print(f"IN_COLAB={IN_COLAB}\nROOT={ROOT}")

installing mlflow dagshub
IN_COLAB=True
ROOT=/content/drive/MyDrive/MLFinalAssignment


In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='smama23', repo_name='MLFinalProject', mlflow=True)

EXPERIMENT_NAME = 'Model_Inference'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)

if mlflow.active_run() is not None:
    mlflow.end_run()

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment  :', EXPERIMENT_NAME)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a2ee46aa-c474-4720-a472-95a7b3b33298&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=f31231d8c06bbed7c6a532de1099a323bfc2f273d060185b80d4158baa38549f




Accessing as smama23

Initialized MLflow to track repo "smama23/MLFinalProject"

Repository smama23/MLFinalProject initialized!

2026/07/11 13:12:03 INFO mlflow.tracking.fluent: Experiment with name 'Model_Inference' does not exist. Creating a new experiment.


Tracking URI: https://dagshub.com/smama23/MLFinalProject.mlflow
Experiment  : Model_Inference


In [4]:
import numpy as np
import pandas as pd

import mlflow
import mlflow.pyfunc
from mlflow import MlflowClient

from walmart_prep import load_raw, make_submission, wmae, wmae_weights, seasonal_naive, FOLDS

EXPERIMENT = EXPERIMENT_NAME
REGISTERED_MODEL_NAME = "WalmartSalesForecast"
client = MlflowClient()
train, test, features, stores = load_raw(str(ROOT))
print("tracking:", mlflow.get_tracking_uri())
print("test:", test.shape)

tracking: https://dagshub.com/smama23/MLFinalProject.mlflow
test: (115064, 4)


In [ ]:
def recent_wmae_of_run(run_id):
    run = client.get_run(run_id)
    m = run.data.metrics
    for k in ("cv_wmae_recent", "wmae_recent", "best_wmae_recent"):
        if k in m:
            return m[k]
    for r in client.search_runs([run.info.experiment_id], order_by=["start_time DESC"],
                                max_results=50):
        for k in ("wmae_recent", "cv_wmae_recent", "best_wmae_recent"):
            if k in r.data.metrics:
                return r.data.metrics[k]
    return None

def gate_of_run(run_id):
    return client.get_run(run_id).data.metrics.get("december_gate_passed")

rm = client.get_registered_model(REGISTERED_MODEL_NAME)
ver_aliases = {}
for a, ver in rm.aliases.items():
    ver_aliases.setdefault(str(ver), []).append(a)

rows = []
for v in client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'"):
    rows.append({
        "version": int(v.version),
        "aliases": ", ".join(ver_aliases.get(str(v.version), [])) or "-",
        "recent_wmae": recent_wmae_of_run(v.run_id),
        "december_gate": gate_of_run(v.run_id),
        "run_id": v.run_id[:8],
    })
board = pd.DataFrame(rows).sort_values(
    "recent_wmae", key=lambda s: s.fillna(1e9)).reset_index(drop=True)
board

,version,aliases,recent_wmae,december_gate,run_id
0,3,xgboost,1616.667340,1.0,29b61d37
1,2,-,1650.262174,1.0,8b17e318
2,1,-,1650.262174,1.0,d1916a0b
3,6,timesfm,1679.105071,0.0,17c9c320
4,5,arima,1791.704456,0.0,26efce9d
5,4,dlinear,1816.221065,0.0,71032c12


In [ ]:
CHAMPION_ALIAS = None

scored = board[board.recent_wmae.notna()]
if CHAMPION_ALIAS is None:
    best = scored.iloc[0]
    champion_version = int(best.version)
    champion_alias = best.aliases.split(",")[0].strip()
    if champion_alias in ("-", ""):
        champion_alias = "champion_src"
        client.set_registered_model_alias(REGISTERED_MODEL_NAME, champion_alias, champion_version)
else:
    champion_alias = CHAMPION_ALIAS
    champion_version = int(client.get_model_version_by_alias(REGISTERED_MODEL_NAME, champion_alias).version)

print(f"champion: version {champion_version}  alias '{champion_alias}'  "
      f"recent WMAE {board.loc[board.version==champion_version,'recent_wmae'].iloc[0]:.1f}")

champion: version 3  alias 'xgboost'  recent WMAE 1616.7


In [7]:
model_uri = f"models:/{REGISTERED_MODEL_NAME}@{champion_alias}"
print("loading", model_uri)
champion = mlflow.pyfunc.load_model(model_uri)

y_pred = np.asarray(champion.predict(test))
print(f"predictions: n={len(y_pred)}  mean={y_pred.mean():.1f}  "
      f"min={y_pred.min():.1f}  max={y_pred.max():.1f}  finite={np.isfinite(y_pred).all()}")
assert len(y_pred) == len(test) and np.isfinite(y_pred).all()

loading models:/WalmartSalesForecast@xgboost


predictions: n=115064  mean=16651.5  min=-3881.6  max=628804.9  finite=True


In [8]:
sub = make_submission(test, y_pred, ROOT / "submissions" / "final_submission.csv")
sample = pd.read_csv(ROOT / "sampleSubmission.csv")
assert list(sub.columns) == ["Id", "Weekly_Sales"]
assert len(sub) == len(sample)
assert (sub.Id.values == sample.Id.values).all(), "Id order must match sampleSubmission.csv"
print("submission rows:", len(sub))
print("wrote", ROOT / "submissions" / "final_submission.csv")
sub.head()

submission rows: 115064
wrote /content/drive/MyDrive/MLFinalAssignment/submissions/final_submission.csv


,Id,Weekly_Sales
0,1_1_2012-11-02,38229.738159
1,1_1_2012-11-09,21174.481201
2,1_1_2012-11-16,20299.196655
3,1_1_2012-11-23,21254.922119
4,1_1_2012-11-30,25428.642700


In [9]:
mlflow.set_experiment(EXPERIMENT)
client.set_registered_model_alias(REGISTERED_MODEL_NAME, "champion", champion_version)
print(f"promoted models:/{REGISTERED_MODEL_NAME}@champion -> version {champion_version}")

with mlflow.start_run(run_name="final_inference"):
    mlflow.log_param("champion_alias", champion_alias)
    mlflow.log_param("champion_version", champion_version)
    mlflow.log_param("model_uri", model_uri)
    mlflow.log_metric("pred_mean", float(y_pred.mean()))
    mlflow.log_metric("pred_min", float(y_pred.min()))
    mlflow.log_metric("pred_max", float(y_pred.max()))
    mlflow.log_metric("n_predictions", len(y_pred))
    mlflow.log_artifact(str(ROOT / "submissions" / "final_submission.csv"))
    board_path = ROOT / "docs" / "registry_leaderboard.csv"
    board.to_csv(board_path, index=False)
    mlflow.log_artifact(str(board_path))
print("logged inference run")

promoted models:/WalmartSalesForecast@champion -> version 3
🏃 View run final_inference at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/5/runs/b84314d625be4a398006a518d836c77c
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/5
logged inference run
